# Construcción de dataset multimodal (clínico + imágenes)

Este notebook genera datasets **multimodales sintéticos** combinando:

- Datos clínicos tabulares (`clinical_features_train/valid/test.csv`).
- Imágenes dermatológicas organizadas por clases en `../datasets/train`, `../datasets/valid`, `../datasets/test`.

La idea es asociar a cada fila clínica una imagen coherente con su etiqueta binaria:

- `target_derm == 0` → se asigna una imagen aleatoria de la clase `Healthy` del split correspondiente.
- `target_derm == 1` → se asigna una imagen aleatoria de alguna clase enferma (`Dermatitis`, `Fungal_infections`, `Hypersensitivity`, `demodicosis`, `ringworm`).

> ⚠️ Importante: este dataset multimodal es **artificial**. Los datos clínicos y las imágenes provienen de datasets independientes de Kaggle y no corresponden al mismo paciente real. El objetivo es exclusivamente académico: **demostrar la arquitectura multimodal y el pipeline de modelado**, no extraer conclusiones clínicas reales.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import random
import os

# Fijar semilla para reproducibilidad
semilla = 42
random.seed(semilla)
np.random.seed(semilla)

# Este notebook se asume ubicado en: SPTMV/Modelado/Build_multimodal_dataset.ipynb
ruta_proyecto = Path("..").resolve()
ruta_datasets = ruta_proyecto / "datasets"

print("Ruta proyecto:", ruta_proyecto)
print("Ruta datasets:", ruta_datasets)

# Rutas de datasets clínicos
ruta_train_clinico = ruta_datasets / "clinical_features_train.csv"
ruta_valid_clinico = ruta_datasets / "clinical_features_valid.csv"
ruta_test_clinico  = ruta_datasets / "clinical_features_test.csv"

for ruta in [ruta_train_clinico, ruta_valid_clinico, ruta_test_clinico]:
    print(ruta, "-> existe:", ruta.exists())

Ruta proyecto: C:\Users\Usuario\Desktop\ESTUDIOS  ARCHIVOS\UPC CICLO 8\MACHINE LEARNING\ML Trabajo Final\SPTMV
Ruta datasets: C:\Users\Usuario\Desktop\ESTUDIOS  ARCHIVOS\UPC CICLO 8\MACHINE LEARNING\ML Trabajo Final\SPTMV\datasets
C:\Users\Usuario\Desktop\ESTUDIOS  ARCHIVOS\UPC CICLO 8\MACHINE LEARNING\ML Trabajo Final\SPTMV\datasets\clinical_features_train.csv -> existe: True
C:\Users\Usuario\Desktop\ESTUDIOS  ARCHIVOS\UPC CICLO 8\MACHINE LEARNING\ML Trabajo Final\SPTMV\datasets\clinical_features_valid.csv -> existe: True
C:\Users\Usuario\Desktop\ESTUDIOS  ARCHIVOS\UPC CICLO 8\MACHINE LEARNING\ML Trabajo Final\SPTMV\datasets\clinical_features_test.csv -> existe: True


In [2]:
def listar_imagenes_por_estado(ruta_split):
    """
    Devuelve dos listas de rutas de imágenes:
    - imagenes_saludables: clase 'Healthy'
    - imagenes_enfermas: clases dermatológicas enfermas
    """
    clases_saludables = ["Healthy"]
    clases_enfermas = ["Dermatitis", "Fungal_infections", "Hypersensitivity", "demodicosis", "ringworm"]
    extensiones_validas = {".jpg", ".jpeg", ".png", ".bmp", ".JPG", ".JPEG", ".PNG", ".BMP"}

    imagenes_saludables = []
    imagenes_enfermas = []

    for clase in clases_saludables:
        carpeta_clase = ruta_split / clase
        if carpeta_clase.exists():
            for raiz, _, archivos in os.walk(carpeta_clase):
                for nombre in archivos:
                    if Path(nombre).suffix in extensiones_validas:
                        imagenes_saludables.append(Path(raiz) / nombre)

    for clase in clases_enfermas:
        carpeta_clase = ruta_split / clase
        if carpeta_clase.exists():
            for raiz, _, archivos in os.walk(carpeta_clase):
                for nombre in archivos:
                    if Path(nombre).suffix in extensiones_validas:
                        imagenes_enfermas.append(Path(raiz) / nombre)

    print(f"Split: {ruta_split.name}")
    print(f"  Imágenes saludables (Healthy): {len(imagenes_saludables)}")
    print(f"  Imágenes enfermas: {len(imagenes_enfermas)}")    

    if not imagenes_saludables:
        raise RuntimeError(f"No se encontraron imágenes 'Healthy' en {ruta_split}")
    if not imagenes_enfermas:
        raise RuntimeError(f"No se encontraron imágenes enfermas en {ruta_split}")

    return imagenes_saludables, imagenes_enfermas


# Probar listado de imágenes para cada split de imágenes
ruta_train_img = ruta_datasets / "train"
ruta_valid_img = ruta_datasets / "valid"
ruta_test_img  = ruta_datasets / "test"

_ = listar_imagenes_por_estado(ruta_train_img)
_ = listar_imagenes_por_estado(ruta_valid_img)
_ = listar_imagenes_por_estado(ruta_test_img)

Split: train
  Imágenes saludables (Healthy): 492
  Imágenes enfermas: 2530
Split: valid
  Imágenes saludables (Healthy): 139
  Imágenes enfermas: 721
Split: test
  Imágenes saludables (Healthy): 69
  Imágenes enfermas: 364


In [3]:
def construir_multimodal_split(nombre_split, ruta_csv_clinico):
    """
    Construye un CSV multimodal para un split (train/valid/test):
    - Carga el CSV clínico.
    - Asigna una ruta de imagen coherente con target_derm.
    - Guarda un nuevo CSV multimodal en datasets/multimodal_<split>.csv.
    """
    print(f"\nConstruyendo dataset multimodal para split: {nombre_split}")
    df = pd.read_csv(ruta_csv_clinico)
    print("Tamaño dataset clínico:", df.shape)

    if "target_derm" not in df.columns:
        raise ValueError("La columna 'target_derm' no se encuentra en el CSV clínico.")

    ruta_split_img = ruta_datasets / nombre_split
    imagenes_saludables, imagenes_enfermas = listar_imagenes_por_estado(ruta_split_img)

    rutas_relativas = []
    for etiqueta in df["target_derm"].values:
        if etiqueta == 0:
            ruta_img = random.choice(imagenes_saludables)
        else:
            ruta_img = random.choice(imagenes_enfermas)

        # Guardamos la ruta relativa respecto a carpeta datasets
        ruta_relativa = ruta_img.relative_to(ruta_datasets)
        rutas_relativas.append(str(ruta_relativa))

    df["ruta_imagen"] = rutas_relativas

    ruta_salida = ruta_datasets / f"multimodal_{nombre_split}.csv"
    df.to_csv(ruta_salida, index=False)
    print("Guardado:", ruta_salida, "| tamaño:", df.shape)

    return df


# Construir datasets multimodales para train, valid y test
df_multimodal_train = construir_multimodal_split("train", ruta_train_clinico)
df_multimodal_valid = construir_multimodal_split("valid", ruta_valid_clinico)
df_multimodal_test  = construir_multimodal_split("test", ruta_test_clinico)


Construyendo dataset multimodal para split: train
Tamaño dataset clínico: (3509, 73)
Split: train
  Imágenes saludables (Healthy): 492
  Imágenes enfermas: 2530
Guardado: C:\Users\Usuario\Desktop\ESTUDIOS  ARCHIVOS\UPC CICLO 8\MACHINE LEARNING\ML Trabajo Final\SPTMV\datasets\multimodal_train.csv | tamaño: (3509, 74)

Construyendo dataset multimodal para split: valid
Tamaño dataset clínico: (752, 73)
Split: valid
  Imágenes saludables (Healthy): 139
  Imágenes enfermas: 721
Guardado: C:\Users\Usuario\Desktop\ESTUDIOS  ARCHIVOS\UPC CICLO 8\MACHINE LEARNING\ML Trabajo Final\SPTMV\datasets\multimodal_valid.csv | tamaño: (752, 74)

Construyendo dataset multimodal para split: test
Tamaño dataset clínico: (752, 73)
Split: test
  Imágenes saludables (Healthy): 69
  Imágenes enfermas: 364
Guardado: C:\Users\Usuario\Desktop\ESTUDIOS  ARCHIVOS\UPC CICLO 8\MACHINE LEARNING\ML Trabajo Final\SPTMV\datasets\multimodal_test.csv | tamaño: (752, 74)


In [4]:
print("\nEjemplo de filas del dataset multimodal (train):")
df_multimodal_train.head()


Ejemplo de filas del dataset multimodal (train):


,Age,Weight_kg,target_derm,age_squared,weight_zscore_by_breed,age_weight_interaction,breed_is_other,is_vaccinated,has_parasite_history,has_chronic_illness,...,breed_grouped_golden retriever,breed_grouped_labrador retriever,breed_grouped_mixed breed,breed_grouped_poodle,breed_grouped_rottweiler,breed_grouped_yorkshire terrier,age_bucket_puppy,age_bucket_adult,age_bucket_senior,ruta_imagen
0,5.8,22.9,1,33.64,-1.787272,132.82,0,0,0,0,...,False,False,False,False,True,False,False,True,False,train\Dermatitis\May-is-allergy-awareness-mont...
1,9.4,3.8,1,88.36,-1.174947,35.72,0,1,0,0,...,False,False,False,True,False,False,False,False,True,train\Dermatitis\1000011971_x4_jpg.rf.8cc46bae...
2,1.8,4.4,1,3.24,-0.930756,7.92,0,0,0,0,...,False,False,False,False,False,True,False,True,False,train\Hypersensitivity\Hypersensitivity_2_jpg....
3,8.2,4.4,1,67.24,-0.930756,36.08,0,0,0,0,...,False,False,False,False,False,True,False,False,True,train\Hypersensitivity\B9781416056638000070_f0...
4,14.6,8.0,0,213.16,0.919236,116.80,0,0,0,0,...,False,False,False,False,False,True,False,False,True,train\Healthy\dogs_084_jpg.rf.8091ecac65c89d1d...
